# 🎥 AI 영상 생성 — 텍스트 한 줄로 영상을 만듭니다

텍스트 설명이나 이미지를 입력하면 AI가 짧은 영상을 만들어줍니다.
음악과 별도로 생성한 뒤, 나중에 편집 소프트웨어에서 합치면 뮤직비디오가 됩니다.

아래 세 가지 도구 중 **원하는 것만 골라서** 실행하세요. 전부 할 필요는 없습니다.

| 도구 | 입력 | 출력 | 특징 |
|------|------|------|------|
| **AnimateDiff** | 이미지 + 텍스트 | 2~4초 영상 | 포스터/커버를 움직이게 |
| **CogVideoX** | 텍스트만 | 4~6초 영상 | 높은 품질, 느림 |
| **Open-Sora** | 텍스트만 | 영상 | 오픈소스 Sora 대안 |

▶ 먼저 공통 패키지를 설치합니다.

In [ ]:
%%capture
!pip install -q diffusers transformers accelerate imageio

import warnings
warnings.filterwarnings('ignore')

print('✅ 공통 패키지 설치 완료!')

---
# 옵션 A: AnimateDiff — 텍스트로 움직이는 영상

텍스트 설명을 넣으면 짧은 영상 (2~4초)이 만들어집니다.
"어두운 콘서트홀에서 카메라가 다가가는" 같은 움직임을 표현할 수 있습니다.

⏰ 생성에 3~5분 소요

### A-1. AnimateDiff 추가 설치 + 모델 로드

▶ 아래 셀을 실행하세요.

In [ ]:
import torch
from diffusers import AnimateDiffPipeline, DDIMScheduler, MotionAdapter
from diffusers.utils import export_to_gif

print('🔄 AnimateDiff 모델을 불러오는 중...')

# v1-5-3: 최신 motion adapter
adapter = MotionAdapter.from_pretrained(
    'guoyww/animatediff-motion-adapter-v1-5-3',
    torch_dtype=torch.float16
)

pipe = AnimateDiffPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    motion_adapter=adapter,
    torch_dtype=torch.float16,
    safety_checker=None
)

pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe.enable_model_cpu_offload()

print('✅ AnimateDiff (motion adapter v1-5-3) 로딩 완료!')

### A-2. 프롬프트 입력 → 영상 생성

▶ 어떤 영상을 만들고 싶은지 영어로 적고 실행하세요.

In [ ]:
import IPython.display as ipd
import os

prompt = "a grand piano in a dark concert hall, camera slowly zooming in, cinematic lighting, 4k"  # ← 여기를 수정하세요
negative_prompt = "blurry, low quality, text, watermark"

print(f'🎬 프롬프트: "{prompt}"')
print('🔄 영상을 생성하고 있습니다... (3~5분 소요)')

output = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_frames=16,
    num_inference_steps=20,
    guidance_scale=7.5,
    width=512, height=512,
)

frames = output.frames[0]
output_gif = 'animatediff_output.gif'
export_to_gif(frames, output_gif)

print('✅ 생성 완료!')
ipd.display(ipd.Image(filename=output_gif))

In [ ]:
from google.colab import files
if os.path.exists(output_gif):
    files.download(output_gif)

---
# 옵션 B: CogVideoX — 텍스트만으로 영상 생성

이미지 없이 **텍스트만으로** 영상을 만듭니다.
"어두운 콘서트홀에서 피아노를 연주하는 장면"이라고 하면 그런 영상이 생성됩니다.

⚠️ 모델이 크므로 T4에서 느릴 수 있습니다. 안 되면 Colab Pro가 필요합니다.

⏰ 생성에 5~10분 소요

### B-1. CogVideoX 설치 + 모델 로드

▶ 아래 셀을 실행하세요. 모델 다운로드에 시간이 걸립니다.

In [ ]:
import torch
import gc

# 이전 모델 메모리 해제
if 'pipe' in dir():
    del pipe
    gc.collect()
    torch.cuda.empty_cache()

print('🔄 CogVideoX-2b 모델을 불러오는 중...')
print('⚠️ T4 GPU에서 메모리가 부족할 수 있습니다.')

try:
    from diffusers import CogVideoXPipeline
    from diffusers.utils import export_to_video

    pipe_cog = CogVideoXPipeline.from_pretrained(
        'THUDM/CogVideoX-2b',
        torch_dtype=torch.float16,
    )
    # T4 최적화: sequential offload + VAE slicing/tiling
    pipe_cog.enable_sequential_cpu_offload()
    pipe_cog.vae.enable_slicing()
    pipe_cog.vae.enable_tiling()

    print('✅ CogVideoX-2b 모델 로딩 완료!')
    cogvideo_available = True
except Exception as e:
    print(f'⚠️ CogVideoX를 로드할 수 없습니다: {e}')
    print('💡 이 모델은 더 큰 GPU가 필요합니다. Colab Pro(A100)를 사용하거나, AnimateDiff를 시도해보세요.')
    cogvideo_available = False

### B-2. 프롬프트 입력 → 영상 생성

▶ 만들고 싶은 영상을 영어로 설명하세요.

In [ ]:
if cogvideo_available:
    from diffusers.utils import export_to_video

    prompt_cog = "A grand piano in a dark concert hall, camera slowly zooming in, cinematic lighting"  # ← 여기를 수정하세요

    print(f'🎬 프롬프트: "{prompt_cog}"')
    print('🔄 영상을 생성하고 있습니다... (5~10분 소요)')

    video = pipe_cog(
        prompt=prompt_cog,
        num_videos_per_prompt=1,
        num_inference_steps=30,
        num_frames=24,
        guidance_scale=6,
    ).frames[0]

    output_cog = 'cogvideo_output.mp4'
    export_to_video(video, output_cog, fps=8)

    print('✅ 생성 완료!')

    from google.colab import files
    files.download(output_cog)
else:
    print('⚠️ CogVideoX가 로드되지 않아 건너뜁니다.')

---
# 옵션 C: Open-Sora — 오픈소스 영상 생성

OpenAI의 Sora와 비슷한 컨셉의 오픈소스 모델입니다.
텍스트만으로 영상을 생성합니다.

⚠️ T4 호환 여부에 따라 실행이 안 될 수 있습니다.
안 되는 경우 위의 AnimateDiff를 사용하세요.

In [ ]:
print('ℹ️ Open-Sora는 설치와 실행이 복잡하고 T4에서 메모리 부족이 발생할 수 있습니다.')
print()
print('💡 대안으로 아래 방법을 추천합니다:')
print('  1. 위의 AnimateDiff (옵션 A) — T4에서 안정적으로 동작')
print('  2. CogVideoX (옵션 B) — Colab Pro에서 사용 가능')
print('  3. Runway Gen-3 (https://runwayml.com) — 유료이지만 최고 품질')
print('  4. Kling (https://klingai.com) — 무료 크레딧 제공')
print()
print('Open-Sora 공식 데모: https://github.com/hpcaitech/Open-Sora')